# 🥋🏃💃 flow3 · LAFAN 手挑窗口技能库 / hand-picked window skill library

每个 cell 点开 = GR00T(checkpoint-6000)预测 token → 注入 SONIC WBC → G1 在 Isaac viewer 动。
**离线回放(无 server,不卡 GUI)**。逐个看哪个保真度好 → 留进 flow3,差的不编入。


## 用法 / How to use
- 点一个 cell 起 viewer 看那个技能(**实时闭环 live**,GR00T checkpoint-6000 逐帧推理,**零回放**);看完关窗口(或跑最后的 `停止` cell)再点下一个。
- viewer:`F`=自由视角 `V`=绿色参考骨架(目标) `C`=贴近机器人 `R`=重置重播。
- server 在各格间复用(:5555),首格起约 1min 载 VLM;切下一格秒进。
- `没摔 ≠ 像那个动作`:这是**视觉保真度筛子**。像 → 留;只是站着晃 → 弃。
- ⚠️ 零回放下每个窗口都是 VLA 从静止自启的 one-shot,可能起不来/趋向站立(单帧 no-memory 限制)—— 这是模型的诚实 live 能力。要「保证流畅」看 `gear_sonic_flow3.sh` 离线版(那是回放,非 live)。
- 预测 token dump(`datasets/sonic_vla_pred_flow3/`)**live 模式不需要**;仅当某段重新开 bootstrap 或跑离线版时才需(下方下载/dump cell)。

## ⬇️ 没自己训练?一键下载已发布模型 / one-click download of the published checkpoint

推理 cell 会**自动判断**:`outputs/gr00t_sonic_flow3/checkpoint-6000` 存在就用自己训练的,
否则回退到下面下载的 [`wsagi/GR00T-N1.7-G1-SONIC-LAFAN`](https://huggingface.co/wsagi/GR00T-N1.7-G1-SONIC-LAFAN)
(默认 HF cache `~/.cache/huggingface/hub`)。

> 离线注入 cell 用的预测 token npz(`datasets/sonic_vla_pred_flow3/`)是 gitignored 产物 ——
> 没有的话先跑下面第二个 cell 用(下载的)模型 dump 一份。数据集同样可从
> [`wsagi/SONIC-VLA-LAFAN`](https://huggingface.co/datasets/wsagi/SONIC-VLA-LAFAN) 下载。


In [ ]:
# 下载模型到默认 HF cache(已在 cache 则秒过,幂等)
!dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-LAFAN'))"


In [ ]:
# (可选)用解析到的 ckpt 重新 dump 预测 token npz -> datasets/sonic_vla_pred_flow3/
# 数据集若缺失: huggingface-cli download wsagi/SONIC-VLA-LAFAN --repo-type dataset --local-dir datasets/sonic_vla_lerobot_flow3
!CKPT=$PWD/outputs/gr00t_sonic_flow3/checkpoint-6000; [ -d "$CKPT" ] || CKPT=$(dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-LAFAN', local_files_only=True))"); \
 cd dependencies/Isaac-GR00T && COMPILE_ACTION_HEAD_DISABLE=1 .venv/bin/python ../../scripts/gr00t_dump_pred_tokens.py \
   --model_path "$CKPT" --dataset_path ../../datasets/sonic_vla_lerobot_flow3 --out ../../datasets/sonic_vla_pred_flow3 \
   --traj_ids 0 1 2 3 4 5 6 7 --keys dance_moonwalk dance_spin_stepback_clap fight_block_pushkick_shove fight_combat_combo_kicks fight_fierce_swings run_circle run_jog_backward run_sprint_backpedal


---
## 🔁 flow3 循环 demo / looping sequence
顺序:**防守-蹬腿 → 转圈跑 → 搏击-连踢 → 快跑-倒跑 → 太空漫步 → 慢跑-倒跑**(≈90s/圈,一直循环)。
拼接 6 个窗口的预测 token 离线注入(无 server 不卡)。改顺序:`SEQ=combat,circle,moonwalk bash ...`。


In [ ]:
!bash scripts/gear_sonic_flow3.sh

### 🛰️ flow3 实时推理版(带 server)/ live closed-loop
**真正的 GR00T 实时推理**(server 在环,非回放):每个窗口短 bootstrap 进动作后交 live GR00T 出 token。
- 同序列 ≈91s/圈循环。第一次起 server 载 VLM ~1min。
- ⚠️ 窗口都是一次性动作,交给 live 后可能**渐渐站住**(无记忆限制)——上面**离线 loop 才是保证流畅**的版本。
- 看完务必跑下面 `停止` cell 关 server+viewer 释放显存。


In [ ]:
# GR00T_CKPT 自动解析:优先 outputs/ 自训,否则用 HF cache 里下载的发布版
!CKPT=$PWD/outputs/gr00t_sonic_flow3/checkpoint-6000; [ -d "$CKPT" ] || CKPT=$(dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-LAFAN', local_files_only=True))" 2>/dev/null) || { echo '先跑上面的下载 cell'; exit 1; }; \
 GR00T_CKPT=$CKPT PKL=data/seg_flow3_all.pkl PRED_DIR=$PWD/datasets/sonic_vla_pred_flow3 bash scripts/gear_sonic_live_demo.sh @flow3


### 🛰️ flow3 实时推理 · StarVLA-CE 版 / live closed-loop, StarVLA per-dim-CE head

同一条 live 链路换 **StarVLA(Qwen3.5-4B 冻结 + QwenPI_CE 交叉熵 head)** 当 token 生产者:
server 起在 :5556,Isaac 侧机制(timeline/bootstrap/settle)完全复用。
开环 A/B:CE 0.0174(8/8 破模板) vs stock flow head 0.0374(背模板) vs GR00T 0.0026 ——
详见 `doc/sonic_starvla_swap_brainstorm.html` §11。预期观感介于 stock(生硬)与 GR00T 之间。


In [ ]:
!bash scripts/starvla_sonic_live_demo.sh @flow3


### 🧱 flow3 实时推理 · MaskBeT 版 / live closed-loop, from-scratch 25M masked transformer

同一条 live 链路再换 **MaskBeT（无 backbone、从零 25M masked transformer，MaskGIT 并行解码）** 当 token 生产者：
server 起在 :5557，Isaac 侧机制（timeline/bootstrap/settle）完全复用。路线 B（LeSONIC/MaskBeT submodule）。

开环 A/B（flow3 8 窗口 MSE-64，同记忆口径）：**MaskBeT 25M 从零 = 0.0090 (expected) / 0.0193 (argmax)** · StarVLA CE v1（4B 冻结 VLM）= 0.0125 (expected) / 0.0174 (argmax) · GR00T = 0.0026 · 模板 = 0.0367。
**25M 从零 ≈/优于 4B 冻结 VLM** —— 瓶颈是数据不是骨干。详见 `MaskBeT/doc/maskbet_design.html` §5.1。

默认 `argmax` 解码（构造性在格，iterative argmax 保 token std ≈ GT 0.179，闭环幅度比 CE 足）；`SONIC_MASKBET_DECODE=expected` 切到 0.0090 那档（开环 MSE 最优，离格，server 内 snap 回格）。
停止：`pkill -f serve_maskbet_sonic`。

In [ ]:
!bash scripts/maskbet_sonic_live_demo.sh @flow3

---
## 🥋 FIGHT 武术


### 🥋 `combat` — *combat strikes and combo kicks*
完整 / full · 物理有效不摔


In [ ]:
# 🥋 combat — GR00T-N1.7 flow3 实时闭环 live(checkpoint-6000, 零回放)
!CKPT=$PWD/outputs/gr00t_sonic_flow3/checkpoint-6000; [ -d "$CKPT" ] || CKPT=$(dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-LAFAN', local_files_only=True))" 2>/dev/null) || { echo '先跑上面的下载 cell'; exit 1; }; \
 GR00T_CKPT=$CKPT PKL=data/seg_flow3_all.pkl BOOTSTRAP_STEPS=0 bash scripts/gear_sonic_live_demo.sh combat

### 🥋 `block` — *block and push-kick*
前缀 72% / prefix (full window falls @72%)


In [ ]:
# 🥋 block — GR00T-N1.7 flow3 实时闭环 live(checkpoint-6000, 零回放)
!CKPT=$PWD/outputs/gr00t_sonic_flow3/checkpoint-6000; [ -d "$CKPT" ] || CKPT=$(dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-LAFAN', local_files_only=True))" 2>/dev/null) || { echo '先跑上面的下载 cell'; exit 1; }; \
 GR00T_CKPT=$CKPT PKL=data/seg_flow3_all.pkl BOOTSTRAP_STEPS=0 bash scripts/gear_sonic_live_demo.sh block

### 🥋 `fierce` — *fierce swings*
前缀 32% / short prefix


In [ ]:
# 🥋 fierce — GR00T-N1.7 flow3 实时闭环 live(checkpoint-6000, 零回放)
!CKPT=$PWD/outputs/gr00t_sonic_flow3/checkpoint-6000; [ -d "$CKPT" ] || CKPT=$(dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-LAFAN', local_files_only=True))" 2>/dev/null) || { echo '先跑上面的下载 cell'; exit 1; }; \
 GR00T_CKPT=$CKPT PKL=data/seg_flow3_all.pkl BOOTSTRAP_STEPS=0 bash scripts/gear_sonic_live_demo.sh fierce

---
## 🏃 RUN 移动


### 🏃 `jogback` — *jog forward then run backward*
完整 / full


In [ ]:
# 🏃 jogback — GR00T-N1.7 flow3 实时闭环 live(checkpoint-6000, 零回放)
!CKPT=$PWD/outputs/gr00t_sonic_flow3/checkpoint-6000; [ -d "$CKPT" ] || CKPT=$(dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-LAFAN', local_files_only=True))" 2>/dev/null) || { echo '先跑上面的下载 cell'; exit 1; }; \
 GR00T_CKPT=$CKPT PKL=data/seg_flow3_all.pkl BOOTSTRAP_STEPS=0 bash scripts/gear_sonic_live_demo.sh jogback

### 🏃 `sprint` — *sprint back and forth then backpedal*
完整 / full


In [ ]:
# 🏃 sprint — GR00T-N1.7 flow3 实时闭环 live(checkpoint-6000, 零回放)
!CKPT=$PWD/outputs/gr00t_sonic_flow3/checkpoint-6000; [ -d "$CKPT" ] || CKPT=$(dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-LAFAN', local_files_only=True))" 2>/dev/null) || { echo '先跑上面的下载 cell'; exit 1; }; \
 GR00T_CKPT=$CKPT PKL=data/seg_flow3_all.pkl BOOTSTRAP_STEPS=0 bash scripts/gear_sonic_live_demo.sh sprint

### 🏃 `circle` — *run in a circle*
完整 / full


In [ ]:
# 🏃 circle — GR00T-N1.7 flow3 实时闭环 live(checkpoint-6000, 零回放)
!CKPT=$PWD/outputs/gr00t_sonic_flow3/checkpoint-6000; [ -d "$CKPT" ] || CKPT=$(dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-LAFAN', local_files_only=True))" 2>/dev/null) || { echo '先跑上面的下载 cell'; exit 1; }; \
 GR00T_CKPT=$CKPT PKL=data/seg_flow3_all.pkl BOOTSTRAP_STEPS=0 bash scripts/gear_sonic_live_demo.sh circle

---
## 💃 DANCE 舞蹈


### 💃 `moonwalk` — *moonwalk*
完整 / full · 严格被旋转倾角误杀,实际不摔


In [ ]:
# 💃 moonwalk — GR00T-N1.7 flow3 实时闭环 live(checkpoint-6000, 零回放)
!CKPT=$PWD/outputs/gr00t_sonic_flow3/checkpoint-6000; [ -d "$CKPT" ] || CKPT=$(dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-LAFAN', local_files_only=True))" 2>/dev/null) || { echo '先跑上面的下载 cell'; exit 1; }; \
 GR00T_CKPT=$CKPT PKL=data/seg_flow3_all.pkl BOOTSTRAP_STEPS=0 bash scripts/gear_sonic_live_demo.sh moonwalk

### 💃 `spinclap` — *spin, step back, and clap*
完整 / full · WBC 跟踪保真度最高(mpjpe_g 90mm)


In [ ]:
# 💃 spinclap — GR00T-N1.7 flow3 实时闭环 live(checkpoint-6000, 零回放)
!CKPT=$PWD/outputs/gr00t_sonic_flow3/checkpoint-6000; [ -d "$CKPT" ] || CKPT=$(dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-LAFAN', local_files_only=True))" 2>/dev/null) || { echo '先跑上面的下载 cell'; exit 1; }; \
 GR00T_CKPT=$CKPT PKL=data/seg_flow3_all.pkl BOOTSTRAP_STEPS=0 bash scripts/gear_sonic_live_demo.sh spinclap

---
## ⏹ 停止(关 viewer,释放显存) / stop


In [ ]:
!bash scripts/gear_sonic_stop.sh